# 06-1 · The evaluation harness

| | |
|---|---|
| **What it does** | samples 200 synthetic groups, runs all four selectors and the mediation arms, writes every CSV the paper's figures and tables read |
| **Proves** | RQ1–RQ4, and the ceiling control that makes the mediation result interpretable |
| **Writes** | `figdata/fig1`, `fig2`, `table1`–`table5` |
| **Determinism** | seeded (`SEED=7`); the same seed reproduces the same CSVs byte for byte |

No API key needed. This notebook is the single source of every number in the paper.


In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
os.chdir(_root)   # figdata/ paths below are repo-relative


In [2]:
import nbio
nbio.bootstrap()


repo     : /project/painting/local-projects-to-run/AnacodicAI-Labs/code/ai-in-food/research_fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/local-projects-to-run/AnacodicAI-Labs/code/ai-in-food/research_fair-mediator')

## Step 1 — parameters, stated once


In [3]:
# Every knob the harness has. Changing any of these changes the paper's numbers.
SEED, N_GROUPS   = 7, 200
MAX_ROUNDS, ETA  = 8, 0.5
CONSENSUS_BAR    = 0.45          # on the worst-off diner's satisfaction
LAMBDAS          = [0, 0.25, 0.5, 1, 2, 5, 10, 25, 50, 100]
SELECTORS        = ["average", "least_misery", "most_pleasure", "fair"]
ADVERSARIAL_FRAC = 0.35
print(f"seed={SEED} groups={N_GROUPS} bar={CONSENSUS_BAR} eta={ETA} cap={MAX_ROUNDS}")


seed=7 groups=200 bar=0.45 eta=0.5 cap=8


## Step 2 — generate the groups

35% are built adversarially: a seed member is drawn, then the remaining members
are chosen to maximise taste distance from them. Without these, every selector
agrees on nearly every group and the comparison is vacuous.


In [4]:
groups  = nbio.generate_groups(n_groups=N_GROUPS, seed=SEED, adversarial_frac=ADVERSARIAL_FRAC)
catalog = nbio.load_catalog()
print(f"groups={len(groups)}  adversarial={sum(g['adversarial'] for g in groups)}  catalog={len(catalog)} options")
print("sizes:", sorted({len(g['members']) for g in groups}))


groups=200  adversarial=70  catalog=5 options
sizes: [3, 4, 5, 6]


## Step 3 — reproducibility, asserted not assumed


In [5]:
# Same seed must give identical groups; a different seed must not.
again = nbio.generate_groups(n_groups=N_GROUPS, seed=SEED, adversarial_frac=ADVERSARIAL_FRAC)
other = nbio.generate_groups(n_groups=N_GROUPS, seed=8, adversarial_frac=ADVERSARIAL_FRAC)
ids  = lambda gs: [[m["member_id"] for m in g["members"]] for g in gs]
assert ids(groups) == ids(again),  "same seed produced different groups"
assert ids(groups) != ids(other),  "different seeds produced identical groups"
print("PASS - generation is deterministic under the seed, and seed-sensitive")


PASS - generation is deterministic under the seed, and seed-sensitive


## Step 4 — Table 1: the four selectors


In [6]:
per = {s: [r for r in (nbio.run_group(g, catalog, s, 0.5) for g in groups) if r] for s in SELECTORS}
rows = []
for s in SELECTORS:
    row = {"selector": s}
    for k in ["mean_sat","min_sat","jain","gini","max_envy","coverage"]:
        m, ci, _ = nbio.mean_ci([r[k] for r in per[s]])
        row[k] = round(m,4); row[k+"_ci95"] = round(ci,4)
    row["n_groups"] = len(per[s]); rows.append(row)
nbio.save_figdata("table1_selectors.csv", rows)
nbio.table([{k:v for k,v in r.items() if not k.endswith("_ci95")} for r in rows])


-> wrote figdata/table1_selectors.csv  (4 rows)
selector       mean_sat  min_sat  jain   gini   max_envy  coverage  n_groups
----------------------------------------------------------------------------
average        0.641     0.459    0.955  0.109  0.018     1.000     200     
least_misery   0.634     0.463    0.957  0.106  0.032     1.000     200     
most_pleasure  0.629     0.436    0.944  0.122  0.046     1.000     200     
fair           0.641     0.459    0.955  0.109  0.018     1.000     200     


## Step 5 — is Fair actually different from Average?


In [7]:
# Paired bootstrap: both arms run on the SAME groups, so pair them.
fa = {r["group_id"]: r for r in per["fair"]}; av = {r["group_id"]: r for r in per["average"]}
common = [i for i in fa if i in av]
for metric in ["min_sat", "jain", "mean_sat"]:
    a = [fa[i][metric] for i in common]; b = [av[i][metric] for i in common]
    ma,cia,_ = nbio.mean_ci(a); mb,cib,_ = nbio.mean_ci(b)
    p = nbio.paired_bootstrap_p(a, b, n_boot=5000, seed=SEED)
    print(f"  {metric:9s} fair={ma:.4f}+-{cia:.4f}  avg={mb:.4f}+-{cib:.4f}  delta={ma-mb:+.4f}  p={p:.4f}")


  min_sat   fair=0.4593+-0.0116  avg=0.4590+-0.0116  delta=+0.0003  p=0.0144


  jain      fair=0.9547+-0.0037  avg=0.9546+-0.0037  delta=+0.0001  p=0.0186


  mean_sat  fair=0.6413+-0.0098  avg=0.6413+-0.0098  delta=-0.0000  p=0.0308


## Step 6 — Figure 1: the lambda sweep, and how often it moves the pick


In [8]:
fig1 = []
for lam in LAMBDAS:
    rs = [r for r in (nbio.run_group(g, catalog, "fair", lam) for g in groups) if r]
    fig1.append({"lambda": lam,
                 "mean_sat": round(sum(r["mean_sat"] for r in rs)/len(rs),4),
                 "jain":     round(sum(r["jain"]     for r in rs)/len(rs),4),
                 "min_sat":  round(sum(r["min_sat"]  for r in rs)/len(rs),4),
                 "pick_id":  f"aggregate_over_{len(rs)}_groups"})
nbio.save_figdata("fig1_lambda_sweep.csv", fig1)
nbio.table(fig1, cols=["lambda","mean_sat","jain","min_sat"])


-> wrote figdata/fig1_lambda_sweep.csv  (10 rows)
lambda  mean_sat  jain   min_sat
--------------------------------
0       0.641     0.955  0.459  
0.250   0.641     0.955  0.459  
0.500   0.641     0.955  0.459  
1       0.641     0.955  0.460  
2       0.641     0.955  0.460  
5       0.638     0.957  0.460  
10      0.632     0.958  0.459  
25      0.619     0.959  0.455  
50      0.612     0.959  0.450  
100     0.611     0.959  0.449  


In [9]:
# How many groups does lambda actually change the recommendation for?
moved = 0
for g in groups:
    picks = {lam: (nbio.run_group(g, catalog, "fair", lam) or {}).get("pick_id") for lam in LAMBDAS}
    if len({p for p in picks.values() if p}) > 1: moved += 1
print(f"lambda changes the pick in {moved}/{len(groups)} groups = {100*moved/len(groups):.1f}%")
print("...but see Table 1: the resulting change in MinSat is ~0.0003.")


lambda changes the pick in 132/200 groups = 66.0%
...but see Table 1: the resulting change in MinSat is ~0.0003.


## Step 7 — Figure 2: satisfaction vs fairness, every group and selector


In [10]:
fig2 = [{"group_id": r["group_id"], "selector": s,
         "mean_sat": round(r["mean_sat"],4), "jain": round(r["jain"],4),
         "min_sat": round(r["min_sat"],4)}
        for s in SELECTORS for r in per[s]]
nbio.save_figdata("fig2_sat_vs_fair.csv", fig2)
print(f"{len(fig2)} points = {len(groups)} groups x {len(SELECTORS)} selectors")


-> wrote figdata/fig2_sat_vs_fair.csv  (800 rows)
800 points = 200 groups x 4 selectors


## Step 8 — the mediation arms, with the ceiling control

The ceiling is the fraction of groups for which *any* option clears the bar.
Reporting convergence without it conflates a failing loop with an impossible
instance — that distinction is the whole point of this cell.


In [11]:
pre = [r for r in (nbio.run_group(g, catalog, "fair", 0.5) for g in groups) if r]

def run_arm(use_veto):
    conv, rounds = 0, []
    for r in pre:
        S, ids = r["_S"], r["_member_ids"]
        med = nbio.Mediator(ids, S, learning_rate=ETA, fairness_lambda=0.5)
        cands, done = list(S.keys()), False
        for rnd in range(1, MAX_ROUNDS+1):
            pick, _ = med.recommend(cands)
            if min(S[pick][m] for m in ids) >= CONSENSUS_BAR:
                conv += 1; rounds.append(rnd); done = True; break
            worst = min(ids, key=lambda m: S[pick][m])
            react = "veto" if (use_veto and med.veto_tokens.get(worst,0) > 0) else "strong_dislike"
            med.run_round(cands, {m: (react if m == worst else "accept") for m in ids})
            if not cands: break
        if not done: rounds.append(MAX_ROUNDS)
    m, _, sd = nbio.mean_ci([float(x) for x in rounds])
    return conv, round(m,3), round(sd,3)

ceiling = sum(1 for r in pre if max(min(r["_S"][o][m] for m in r["_member_ids"]) for o in r["_S"]) >= CONSENSUS_BAR)
oneshot = sum(1 for r in pre if min(r["_S"][r["pick_id"]][m] for m in r["_member_ids"]) >= CONSENSUS_BAR)
c_on, r_on, s_on = run_arm(False)
c_vt, r_vt, s_vt = run_arm(True)
n = len(pre)
print(f"  ceiling (any option clears bar): {ceiling}/{n} = {100*ceiling/n:.1f}%")
print(f"  one-shot : {oneshot}/{n}  = {100*oneshot/ceiling:.1f}% of ceiling")
print(f"  mediator : {c_on}/{n}  = {100*c_on/ceiling:.1f}% of ceiling, {r_on:.2f} rounds")
print(f"  +veto    : {c_vt}/{n}  = {100*c_vt/ceiling:.1f}% of ceiling, {r_vt:.2f} rounds")


  ceiling (any option clears bar): 103/200 = 51.5%
  one-shot : 102/200  = 99.0% of ceiling
  mediator : 103/200  = 100.0% of ceiling, 4.41 rounds
  +veto    : 103/200  = 100.0% of ceiling, 4.40 rounds


In [12]:
nbio.save_figdata("table2_mediator.csv", [
 {"arm":"mediator_off (one-shot)","rounds_mean":1.0,"rounds_sd":0.0,
  "converged_frac":round(oneshot/n,4),"pct_of_reachable":round(100*oneshot/ceiling,1),"n_groups":n},
 {"arm":"mediator_on","rounds_mean":r_on,"rounds_sd":s_on,
  "converged_frac":round(c_on/n,4),"pct_of_reachable":round(100*c_on/ceiling,1),"n_groups":n},
 {"arm":"ceiling (best option in F)","rounds_mean":"--","rounds_sd":"--",
  "converged_frac":round(ceiling/n,4),"pct_of_reachable":100.0,"n_groups":n}])
nbio.save_figdata("table4_veto.csv", [
 {"arm":"veto_off","rounds_mean":r_on,"rounds_sd":s_on,"converged_frac":round(c_on/n,4),"n_groups":n},
 {"arm":"veto_on", "rounds_mean":r_vt,"rounds_sd":s_vt,"converged_frac":round(c_vt/n,4),"n_groups":n}])


-> wrote figdata/table2_mediator.csv  (3 rows)
-> wrote figdata/table4_veto.csv  (2 rows)


PosixPath('/project/painting/local-projects-to-run/AnacodicAI-Labs/code/ai-in-food/research_fair-mediator/figdata/table4_veto.csv')

## Step 9 — Table 5: dish compensation, and its caveat


In [ ]:
gm = {m["member_id"]: m for g in groups for m in g["members"]}
# One row per assignment level, all over the SAME groups (those where every
# level is defined), so the rows compare like with like. B1-B3 are the
# baselines from nbio (PRD-11): what C4 gains over B1 is compensation; over B2
# is personalisation; the rest is the scale difference flagged below.
levels = [
 ("venue whole-menu average (naive baseline)", None),
 ("B1: random safe dish, per diner",      lambda ms, pk, row: nbio.assign_dishes_random(ms, pk, seed=SEED)),
 ("B3: median-diner's best dish, shared", lambda ms, pk, row: nbio.assign_dishes_median_diner(ms, pk, row)),
 ("B2: best shared dish (one for all)",   lambda ms, pk, row: nbio.assign_dishes_shared_best(ms, pk)),
 ("per-diner best safe dish (C4)",        lambda ms, pk, row: nbio.assign_dishes_best(ms, pk)),
]
mins = {name: [] for name, _ in levels}
means = {name: [] for name, _ in levels}
for r in pre:
    pick = next(o for o in r["_F"] if o["option_id"] == r["pick_id"])
    ms = [gm[mid] for mid in r["_member_ids"]]
    row = r["_S"][r["pick_id"]]
    got = {}
    for name, fn in levels:
        got[name] = dict(row) if fn is None else fn(ms, pick, row)
    if any(v is None for v in got.values()):
        continue
    for name, v in got.items():
        mins[name].append(min(v.values())); means[name].append(sum(v.values()) / len(v))

c4 = "per-diner best safe dish (C4)"
c4_min = sum(mins[c4]) / len(mins[c4])
rows5 = []
for name, _ in levels:
    m, hw, _sd = nbio.mean_ci(mins[name])
    rows5.append({"level": name, "min_sat": round(m, 4), "min_sat_ci95": round(hw, 4),
                  "mean_sat": round(sum(means[name]) / len(means[name]), 4),
                  "delta_vs_C4": "--" if name == c4 else round(c4_min - m, 4),
                  "n_groups": len(mins[name])})
nbio.save_figdata("table5_compensation.csv", rows5)
venue = "venue whole-menu average (naive baseline)"
p5 = nbio.paired_bootstrap_p(mins[c4], mins[venue], n_boot=5000, seed=SEED)
nbio.table(rows5, cols=["level", "min_sat", "min_sat_ci95", "mean_sat", "delta_vs_C4", "n_groups"], width=20)
print(f"MinSat  restaurant={sum(mins[venue])/len(mins[venue]):.4f} -> dish={c4_min:.4f}"
      f"  delta={c4_min-sum(mins[venue])/len(mins[venue]):+.4f}  p={p5:.4f}")
print("CAVEAT: these are not the same scale -- a dish picked to fit one diner will")
print("        nearly always beat a whole-menu average. Upper bound, not effect size.")


## Step 10 — Table 3: generalisation, reported honestly


In [14]:
fair_row = next(r for r in rows if r["selector"] == "fair")
nbio.save_figdata("table3_datasets.csv", [
 {"dataset":"potluck (5 menu-backed restaurants)","selector":"fair",
  "mean_sat":fair_row["mean_sat"],"min_sat":fair_row["min_sat"],
  "jain":fair_row["jain"],"n_groups":fair_row["n_groups"]},
 {"dataset":"NOT YET SELECTED","selector":"--","mean_sat":"--",
  "min_sat":"--","jain":"--","n_groups":0}])
print("Only one dataset evaluated -- the generalisation claim is unsupported.")


-> wrote figdata/table3_datasets.csv  (2 rows)
Only one dataset evaluated -- the generalisation claim is unsupported.


### Reading it

Three of the four hypotheses come back unsupported, and all three share a
cause. The dial changes the selected restaurant in two thirds of groups yet
moves the worst-off diner's satisfaction by $0.0003$; the loop attains
$100\%$ of the achievable ceiling but beats one-shot by one group in two
hundred; the veto changes nothing because the loop already reaches the
ceiling. Five restaurants whose taste vectors agree to within $0.1$ leave an
aggregation rule almost nothing to choose between.

The one large effect — per-diner dish assignment, $+0.39$ MinSat — is
measured across a scale boundary and is an upper bound, not an effect size.

**This harness is unchanged by the fix.** Expand the catalog (PRD-4), re-run
this notebook, and the same ten cells answer the same four questions on data
capable of separating them.
